# 02 – Split and pipeline checks (tasks 16, 17, 19)

Everything that must be fixed before the first model is trained.

**1–4. Split (task 16).** We split the buildings once, and everyone uses this split. Two rules:
- **Groups stay together:** buildings on the same boiler, or twins, have the same value, so they
  must be on the same side. Otherwise a model could predict a building by copying its twin.
- **Houses in their normal share:** houses are only about 9% of the data, so we balance them
  across all parts (feedback point 1).

About 14% of buildings form the test set, used once at the end (task 26). The rest is split into 5 folds.

**5–7. Checks** of the shared code on small made-up examples and on one fold: the metric functions (task 17),
the expert rule and the preprocessing pipeline (task 19).

**Output:** two separate files, so the test set is locked away:
- `data/processed/cv.csv`: the buildings for cross-validation, all columns plus `fold` (0–4). Read with `load_cv()`.
- `data/processed/test.csv`: the test set, all columns. Read with `load_test()`, only in `05_evaluation.ipynb`.

After this notebook, no other notebook reads `building_clean.csv`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LinearRegression

from config import SEED, THRESHOLD, PROCESSED, load_cv  # one seed for the whole project, same split every run
from metrics import share_above_in_top, share_captured, mae, evaluate
from pipeline import split_xy, build_pipeline, ExpertRuleEncoder

TEST_PARTS = 7     # 1 of 7 parts = about 14% test set
N_FOLDS = 5        # folds for cross-validation

data = pd.read_csv(PROCESSED / "building_clean.csv")
print("Buildings:", len(data), "| houses:", data["house"].sum(), "| groups:", data["group_id"].nunique())

## 1. Test set

`StratifiedGroupKFold` splits the data into 7 parts, keeping groups together ("Group") and
the share of houses about equal in every part ("Stratified"). We take one part as the test set.
`shuffle=True` is needed, otherwise the seed has no effect.

In [ ]:
test_splitter = StratifiedGroupKFold(n_splits=TEST_PARTS, shuffle=True, random_state=SEED)
rest_idx, test_idx = next(test_splitter.split(data, y=data["house"], groups=data["group_id"]))  # first of the 7 parts

data["test"] = False
data.loc[data.index[test_idx], "test"] = True
print("Test set:", len(test_idx), "buildings,", data.loc[data["test"], "house"].sum(), "houses")

## 2. Five folds for cross-validation

The remaining buildings are split the same way into 5 folds (0–4). In each round of
cross-validation, one fold is the validation part and the other four are the training part.

In [ ]:
rest = data[data["test"] == False]
data["fold"] = pd.Series(pd.NA, index=data.index, dtype="Int64")  # empty for test buildings

cv_splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, val_idx) in enumerate(cv_splitter.split(rest, y=rest["house"], groups=rest["group_id"])):
    data.loc[rest.index[val_idx], "fold"] = k   # buildings in the k-th validation part get fold k

print("Buildings without a fold (should be the test set):", data["fold"].isna().sum())

## 3. Checks of the split

Every part should have about the same size and house share, and no group may appear in more than one part.
The share of houses above 450 is not balanced on purpose; with about 200 houses per part it varies by a few points.

In [ ]:
data["part"] = data["fold"].astype("string").fillna("test")  # "0"–"4" or "test"

summary = data.groupby("part").agg(
    buildings=("EGID", "size"),
    houses=("house", "sum"),
    groups=("group_id", "nunique"),
)
summary["share of buildings (%)"] = (summary["buildings"] / len(data) * 100).round(1)
summary["house share (%)"] = (summary["houses"] / summary["buildings"] * 100).round(1)
summary["houses above 450 (%)"] = (
    data[data["house"]].groupby("part")["y"].apply(lambda s: (s > THRESHOLD).mean() * 100).round(1)
)
display(summary)

parts_per_group = data.groupby("group_id")["part"].nunique()
print("Groups split across parts:", (parts_per_group > 1).sum())  # must be 0

assert (parts_per_group == 1).all(), "a group appears in more than one part!"
assert data.loc[data["test"], "fold"].isna().all(), "a test building has a fold number!"
assert data.loc[data["test"] == False, "fold"].notna().all(), "a non-test building has no fold!"

## 4. Save: two separate files

The rows keep the order of `building_clean.csv`. This matters: the selection measure breaks ties with a fixed
shuffle of the row positions (`metrics.py`), so the same order gives exactly the same results every run.

In [ ]:
cv = data[data["test"] == False].drop(columns=["test", "part"])
cv["fold"] = cv["fold"].astype(int)
test = data[data["test"]].drop(columns=["test", "fold", "part"])

cv.to_csv(PROCESSED / "cv.csv", index=False)
test.to_csv(PROCESSED / "test.csv", index=False)
print("Saved cv.csv:", len(cv), "buildings | test.csv:", len(test), "buildings")
assert len(cv) + len(test) == len(data) and not set(cv["EGID"]) & set(test["EGID"])

## 5. Check the metric functions (task 17)

We test the functions from our metrics file on 10 made-up houses where the right answers can be counted
by hand. The expected results are written next to each check; the table at the end
shows that `evaluate` returns one row for houses and one for all buildings.

In [ ]:
# 10 made-up houses. Four are truly above 450: positions 0, 1, 2 and 5.
y_true = [600, 500, 470, 300, 200, 480, 350, 250, 400, 100]
score  = [590, 520, 300, 310, 210, 400, 360, 260, 410, 110]

# The top 20% = 2 houses with the highest score: 590 and 520 (positions 0 and 1).
# Both are truly above 450, so the selection measure should be 2/2 = 1.0
print(share_above_in_top(y_true, score, top=0.20))   # expect 1.0

# They capture 2 of the 4 houses above 450: 2/4 = 0.5
print(share_captured(y_true, score, top=0.20))       # expect 0.5

# The top 40% = 4 houses: positions 0, 1, 8 (410), 5 (400).
# Truly above 450: positions 0, 1 and 5, so 3/4 = 0.75
print(share_above_in_top(y_true, score, top=0.40))   # expect 0.75

# MAE: a perfect prediction must give 0
print(mae(y_true, y_true))                           # expect 0.0

# The full table: half "houses", half not
toy_is_house = [True] * 5 + [False] * 5
evaluate(y_true, score, toy_is_house)

## 6. Check the expert rule on a small made-up example (task 19)

6 made-up buildings, overall mean = 2,400 / 6 = **400**. We use a smoothing of m = 4 so the values are easy to count by hand:
value = (n · mean of combination + 4 · 400) / (n + 4).

In [ ]:
toy_X = pd.DataFrame({
    "construction_period": ["A", "A", "A", "A", "B", "C"],
    "energy_group":        ["gas", "gas", "gas", "gas", "oil", "wood"],
})
toy_y = [200, 200, 200, 200, 1000, 600]

rule = ExpertRuleEncoder(smoothing=4).fit(toy_X, toy_y)
print("Overall mean:", rule.global_mean_)        # expect 400
print(rule.rule_values_)
# A|gas:  n=4, mean 200  -> (4*200 + 4*400) / 8 = 300
# B|oil:  n=1, mean 1000 -> (1000 + 4*400) / 5  = 520   (small cell, pulled strongly towards 400)
# C|wood: n=1, mean 600  -> (600 + 4*400) / 5   = 440

new = pd.DataFrame({"construction_period": ["A", "Z"], "energy_group": ["gas", "gas"]})
print(rule.transform(new).ravel())               # expect [300, 400]: unknown combination gets the overall mean

# without smoothing (m = 0) the rule is just the mean of each combination
print(ExpertRuleEncoder(smoothing=0).fit(toy_X, toy_y).rule_values_)  # expect A|gas 200, B|oil 1000, C|wood 600

## 7. Check the pipeline on one fold (leakage check, task 19)

Rule 2: everything that learns from data is fitted on the training part only. We fit the pipeline once on folds 1–4
and apply it to fold 0, then check what it learned. The cross-validation in `04_models.ipynb` does exactly this
for every fold. From here on we only use the cross-validation data (`load_cv()`).

In [ ]:
train = load_cv()
X, y, is_house, groups = split_xy(train)
folds = train["fold"].to_numpy()

check_train, check_val = folds != 0, folds == 0
pipe = build_pipeline(LinearRegression()).fit(X[check_train], y[check_train])

prep = pipe.named_steps["prep"]
X_val_prep = prep.transform(X[check_val])
names = prep.get_feature_names_out()
print("Columns after preprocessing:", X_val_prep.shape[1])
print("Missing values after preprocessing:", np.isnan(X_val_prep).sum())   # expect 0
print("Expert-rule column present:", "expert_rule" in names)               # expect True

# what the expert rule learned must come from folds 1-4 only, not from fold 0 or the test set
rule = prep.named_transformers_["expert_rule"].named_steps["rule"]
print()
print("Overall mean, learned:", round(rule.global_mean_, 2),
      "| folds 1-4:", round(y[check_train].mean(), 2),
      "| all buildings in cross-validation:", round(y.mean(), 2))
print("Buildings counted in the expert rule:", sum(rule.counts_.values()),
      "| buildings in folds 1-4:", check_train.sum())

assert np.isclose(rule.global_mean_, y[check_train].mean()), "the expert rule did not learn from the training part only"
assert sum(rule.counts_.values()) == check_train.sum(), "the expert rule counted buildings outside the training part"